In [1]:
# Install required libraries
!pip install --user streamlit numpy pandas scikit-learn matplotlib seaborn

In [3]:
# (Optional) Set your ngrok authtoken (get it from https://dashboard.ngrok.com/auth)
# !ngrok authtoken YOUR_AUTH_TOKEN  # Replace with your token

In [12]:
#!streamlit run app.py. this is ran through the cmd prompt. 
# 1. Download python 3.x.x when installer's up, make sure you check add python to path. then you're gold!

In [24]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import sys

In [29]:
red_wine = pd.read_csv('winequality-red.csv', sep=';')
white_wine = pd.read_csv('winequality-white.csv', sep=';')
wine_data = pd.concat([red_wine, white_wine], axis=0)

In [30]:
wine_data.head(5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [31]:
wine_data.columns # Check the columns available in the dataframe

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

In [32]:
wine_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6497 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 659.9 KB


In [46]:
%%writefile radj.py
#Importing necessary libraries
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import sys
import warnings

#This must be first streamlit command
st.set_page_config(layout="wide")

#Load wine dataset
@st.cache_data
def load_data():
    red_wine = pd.read_csv('winequality-red.csv', sep=';')
    white_wine = pd.read_csv('winequality-white.csv', sep=';')
    red_wine['color'] = 'red'
    white_wine['color'] = 'white'
    wine_data = pd.concat([red_wine, white_wine], axis=0)
    return wine_data

wine_data = load_data()

st.title("Wine Quality🍷")

# Create tabs
tab1, tab2, tab3, tab4, tab5, tab6, tab7 = st.tabs([
    "Metadata", "Data", "Color", "Quality", "Scatter", "Box", "Classification"
])

# Metadata Tab
with tab1:
    st.header("Wine Quality Dataset Metadata")
    st.markdown("""
    **Dataset Information:**
    - Source: UCI Machine Learning Repository
    - URL: https://archive.ics.uci.edu/dataset/186/wine+quality
    - Two datasets: Red wine and white wine samples from Portugal
    
    **Attributes (all numerical except color):**
    1. fixed acidity
    2. volatile acidity
    3. citric acid
    4. residual sugar
    5. chlorides
    6. free sulfur dioxide
    7. total sulfur dioxide
    8. density
    9. pH
    10. sulphates
    11. alcohol
    12. quality (score between 0 and 10)
    13. color (red or white)
    """)

# Data Tab
with tab2:
    st.header("Wine Data Viewer")
    view_option = st.radio("Select columns to display:", 
                         ("Chemical", "Alcohol", "Quality", "All"),
                         horizontal=True)
    
    if view_option == "Chemical":
        cols = ['color'] + [col for col in wine_data.columns if col not in ['color', 'alcohol', 'quality']]
        st.dataframe(wine_data[cols])
    elif view_option == "Alcohol":
        st.dataframe(wine_data[['color', 'alcohol']])
    elif view_option == "Quality":
        st.dataframe(wine_data[['color', 'quality']])
    else:
        # Reorder columns to put color first
        cols = ['color'] + [col for col in wine_data.columns if col != 'color']
        st.dataframe(wine_data[cols])

# Color Tab
with tab3:
    st.header("Wine Color Distribution")
    color_counts = wine_data['color'].value_counts()
    fig, ax = plt.subplots()
    color_counts.plot(kind='bar', color=['white', 'red'], edgecolor='black', ax=ax)
    ax.set_xlabel('Wine Color')
    ax.set_ylabel('Count')
    ax.set_title('Number of Wines by Color')
    st.pyplot(fig)

with tab4:
    st.header("Wine Quality Distribution")
    quality_counts = wine_data['quality'].value_counts().sort_index()
    
    # Filter out qualities that are outside the range of 3-9
    quality_counts = quality_counts[quality_counts.index.isin(range(3, 10))]
    
    fig, ax = plt.subplots()
    quality_counts.plot(
        kind='bar', 
        color='#1f77b4',  # A vibrant shade of blue
        edgecolor='#1f77b4', 
        width=0.85,  # Adjusted bar width
        ax=ax
    )
    ax.set_xlabel('Wine Quality')
    ax.set_ylabel('Count')
    ax.set_title('Number of Wines by Quality (3-9)')
    st.pyplot(fig)

# Scatter Tab
with tab5:
    st.header("Scatter Plot Comparison")
    col1, col2 = st.columns(2)
    with col1:
        x_axis = st.selectbox("X-axis:", wine_data.columns[:-1], index=0)
    with col2:
        y_axis = st.selectbox("Y-axis:", wine_data.columns[:-1], index=1)
    
    if x_axis and y_axis:
        fig, ax = plt.subplots()
        sns.scatterplot(data=wine_data, x=x_axis, y=y_axis, hue='color', ax=ax)
        ax.set_title(f"{x_axis} vs {y_axis}")
        st.pyplot(fig)

# Box Tab
with tab6:
    st.header("Box Plot Viewer")
    selected_col = st.selectbox("Select a column for box plot:", 
                               wine_data.columns[:-1], index=0)
    
    fig, ax = plt.subplots()
    sns.boxplot(data=wine_data, x='color', y=selected_col, ax=ax)
    ax.set_title(f"Distribution of {selected_col} by Wine Color")
    st.pyplot(fig)

# Classification Tab
with tab7:
    st.header("Wine Quality Classification")
    
    # Prepare data
    X = wine_data.drop(['quality', 'color'], axis=1)
    y = wine_data['quality']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train model
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Show classification report
    st.subheader("Classification Report")
    report = classification_report(y_test, y_pred, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    st.dataframe(report_df)
    
    # Show feature importance
    st.subheader("Feature Importances")
    importances = model.feature_importances_
    features = X.columns
    importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
    importance_df = importance_df.sort_values('Importance', ascending=False)
    
    fig, ax = plt.subplots()
    sns.barplot(data=importance_df, x='Importance', y='Feature', ax=ax)
    ax.set_title("Feature Importance for Quality Prediction")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)  # Rotate x-axis labels
    st.pyplot(fig)

Overwriting radj.py
